# AdS4 Phase-Lock Demo with Wilson Loop Variants (v7)

Colab-ready toy demo extending v6 with a higher-D probe based on Wilson loop variants.

This version compares:
- **base stack**: EE ⊕ WL ⊕ GEO + consistency
- **extended stack**: base stack + WL variants

Goal:
- test whether additional Wilson loop variants improve stability and uniqueness
- preserve the clean phase-lock structure while adding modular-style probe diversity
- visualize bulk slices, losses, and error diagnostics


In [ ]:
import math, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


In [ ]:
# Toy AdS4-like background with two metric functions
r_h = 1.0
ELL_MIN, ELL_MAX = 0.1, math.pi
R_MIN, R_MAX = 1.05, 3.5

def f_true(r):
    return r**2 + 1.0 - (r_h**3 / r)

def g_true(r):
    return 1.0 / (f_true(r) + 1e-6)

def r_star(ell):
    return r_h + 0.4 + 2.6 / (ell + 0.45)

def s_ee_true(ell):
    rs = r_star(ell)
    f = f_true(rs)
    return torch.log(1 + 3.2 * ell) + 0.10 / torch.sqrt(f + 1e-6)

def w_wl_true(ell):
    rs = r_star(ell)
    f = f_true(rs)
    g = g_true(rs)
    return torch.sqrt(ell + 0.25) + 0.14 / (f + 1.8 * g + 0.2)

def geo_true(ell):
    rs = r_star(ell)
    f = f_true(rs)
    g = g_true(rs)
    return torch.log(ell + 1.0) + 0.11 / torch.sqrt(f + 0.7 * g + 1e-6)

def wlv1_true(ell):
    rs = r_star(ell)
    f = f_true(rs)
    g = g_true(rs)
    return torch.log(ell + 1.2) + 0.09 / (f + 2.2 * g + 0.15)

def wlv2_true(ell):
    rs = r_star(ell)
    f = f_true(rs)
    g = g_true(rs)
    return torch.sqrt(ell + 0.35) + 0.07 / torch.sqrt(f + 1.1 * g + 1e-6)

ell = torch.linspace(ELL_MIN, ELL_MAX, 180, device=device).view(-1, 1)
s_obs = s_ee_true(ell)
w_obs = w_wl_true(ell)
geo_obs = geo_true(ell)
wlv1_obs = wlv1_true(ell)
wlv2_obs = wlv2_true(ell)
r_plot = torch.linspace(R_MIN, R_MAX, 300, device=device).view(-1, 1)


In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, width=64, depth=3):
        super().__init__()
        layers = []
        d = in_dim
        for _ in range(depth):
            layers += [nn.Linear(d, width), nn.Tanh()]
            d = width
        layers.append(nn.Linear(d, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class LModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.core = MLP(1, 1, width=32, depth=2)

    def forward(self, ell):
        return r_h + 0.05 + F.softplus(self.core(ell))

class VModelAdS4(nn.Module):
    def __init__(self):
        super().__init__()
        self.f_head = MLP(1, 1, width=64, depth=3)
        self.g_head = MLP(1, 1, width=64, depth=3)

    def forward(self, r):
        f = F.softplus(self.f_head(r))
        g = F.softplus(self.g_head(r))
        return f, g


In [ ]:
def train_model(use_variants=False, epochs=2000, phase_weight=0.02, metric_weight=0.35, consistency_weight=0.15, variant_weight=1.0):
    L = LModel().to(device)
    V = VModelAdS4().to(device)
    optL = torch.optim.Adam(L.parameters(), lr=2e-3)
    optV = torch.optim.Adam(V.parameters(), lr=2e-3)

    history = {'total': [], 'ee': [], 'wl': [], 'geo': [], 'metric': [], 'consistency': [], 'variant': []}

    def phase_lock_penalty(*losses):
        vals = [torch.sqrt(x + 1e-12) for x in losses]
        if len(vals) <= 1:
            return torch.tensor(0.0, device=device)
        ref = vals[0]
        return sum(torch.log(v / (ref + 1e-12))**2 for v in vals[1:])

    for epoch in range(1, epochs + 1):
        optL.zero_grad()
        r_hat = L(ell)
        f_hat, g_hat = V(r_hat)

        s_pred = torch.log(1 + 3.2 * ell) + 0.10 / torch.sqrt(f_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.25) + 0.14 / (f_hat + 1.8 * g_hat + 0.2)
        geo_pred = torch.log(ell + 1.0) + 0.11 / torch.sqrt(f_hat + 0.7 * g_hat + 1e-6)
        wlv1_pred = torch.log(ell + 1.2) + 0.09 / (f_hat + 2.2 * g_hat + 0.15)
        wlv2_pred = torch.sqrt(ell + 0.35) + 0.07 / torch.sqrt(f_hat + 1.1 * g_hat + 1e-6)

        loss_ee_L = ((s_pred - s_obs)**2).mean()
        loss_wl_L = ((w_pred - w_obs)**2).mean()
        loss_geo_L = ((geo_pred - geo_obs)**2).mean()
        loss_variant_L = (((wlv1_pred - wlv1_obs)**2).mean() + ((wlv2_pred - wlv2_obs)**2).mean()) if use_variants else torch.tensor(0.0, device=device)

        active_losses = [loss_ee_L, loss_wl_L, loss_geo_L]
        if use_variants:
            active_losses.append(loss_variant_L)
        loss_phase_L = phase_lock_penalty(*active_losses)

        lossL = loss_ee_L + loss_wl_L + loss_geo_L + (variant_weight * loss_variant_L if use_variants else 0.0) + phase_weight * loss_phase_L
        lossL.backward()
        optL.step()

        optV.zero_grad()
        r_hat = L(ell).detach()
        f_hat, g_hat = V(r_hat)

        s_pred = torch.log(1 + 3.2 * ell) + 0.10 / torch.sqrt(f_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.25) + 0.14 / (f_hat + 1.8 * g_hat + 0.2)
        geo_pred = torch.log(ell + 1.0) + 0.11 / torch.sqrt(f_hat + 0.7 * g_hat + 1e-6)
        wlv1_pred = torch.log(ell + 1.2) + 0.09 / (f_hat + 2.2 * g_hat + 0.15)
        wlv2_pred = torch.sqrt(ell + 0.35) + 0.07 / torch.sqrt(f_hat + 1.1 * g_hat + 1e-6)

        loss_ee_V = ((s_pred - s_obs)**2).mean()
        loss_wl_V = ((w_pred - w_obs)**2).mean()
        loss_geo_V = ((geo_pred - geo_obs)**2).mean()
        loss_variant_V = (((wlv1_pred - wlv1_obs)**2).mean() + ((wlv2_pred - wlv2_obs)**2).mean()) if use_variants else torch.tensor(0.0, device=device)

        active_losses = [loss_ee_V, loss_wl_V, loss_geo_V]
        if use_variants:
            active_losses.append(loss_variant_V)
        loss_phase_V = phase_lock_penalty(*active_losses)

        f_plot_pred, g_plot_pred = V(r_plot)
        metric_loss = ((f_plot_pred - f_true(r_plot))**2).mean() + ((g_plot_pred - g_true(r_plot))**2).mean()
        consistency_loss = ((f_plot_pred * g_plot_pred - 1.0)**2).mean()

        lossV = loss_ee_V + loss_wl_V + loss_geo_V + (variant_weight * loss_variant_V if use_variants else 0.0) + metric_weight * metric_loss + consistency_weight * consistency_loss + phase_weight * loss_phase_V
        lossV.backward()
        optV.step()

        history['total'].append((lossL + lossV).item())
        history['ee'].append(loss_ee_V.item())
        history['wl'].append(loss_wl_V.item())
        history['geo'].append(loss_geo_V.item())
        history['metric'].append(metric_loss.item())
        history['consistency'].append(consistency_loss.item())
        history['variant'].append(loss_variant_V.item() if use_variants else 0.0)

        if epoch % 400 == 0 or epoch == 1:
            mode = 'base' if not use_variants else 'base+variants'
            print(f"{mode} | epoch {epoch:4d} | total={history['total'][-1]:.6f} | metric={history['metric'][-1]:.6f} | variant={history['variant'][-1]:.6f}")

    with torch.no_grad():
        r_hat = L(ell)
        f_hat, g_hat = V(r_hat)
        s_pred = torch.log(1 + 3.2 * ell) + 0.10 / torch.sqrt(f_hat + 1e-6)
        w_pred = torch.sqrt(ell + 0.25) + 0.14 / (f_hat + 1.8 * g_hat + 0.2)
        geo_pred = torch.log(ell + 1.0) + 0.11 / torch.sqrt(f_hat + 0.7 * g_hat + 1e-6)
        wlv1_pred = torch.log(ell + 1.2) + 0.09 / (f_hat + 2.2 * g_hat + 0.15)
        wlv2_pred = torch.sqrt(ell + 0.35) + 0.07 / torch.sqrt(f_hat + 1.1 * g_hat + 1e-6)
        f_pred, g_pred = V(r_plot)

    return {
        'history': history,
        's_pred': s_pred,
        'w_pred': w_pred,
        'geo_pred': geo_pred,
        'wlv1_pred': wlv1_pred,
        'wlv2_pred': wlv2_pred,
        'f_pred': f_pred,
        'g_pred': g_pred,
    }


In [ ]:
run_base = train_model(use_variants=False)
run_ext = train_model(use_variants=True)


In [ ]:
# Bulk-from-slices visualization from extended run
f_vals = run_ext['f_pred'].squeeze().detach().cpu()
r_vals = r_plot.squeeze().detach().cpu()
thetas = torch.linspace(0, 2*math.pi, 90)
R, T = torch.meshgrid(r_vals, thetas, indexing='ij')
radius_embed = torch.sqrt(torch.clamp(f_vals, min=1e-6)).unsqueeze(1)
X = radius_embed * torch.cos(T)
Y = radius_embed * torch.sin(T)
Z = R


In [ ]:
# Main summary figure
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.ravel()

axes[0].plot(r_plot.cpu(), f_true(r_plot).cpu(), label='f true')
axes[0].plot(r_plot.cpu(), g_true(r_plot).cpu(), label='g true')
axes[0].plot(r_plot.cpu(), run_base['f_pred'].cpu(), '--', label='f base')
axes[0].plot(r_plot.cpu(), run_base['g_pred'].cpu(), ':', label='g base')
axes[0].set_title('Base stack')
axes[0].legend(fontsize=8)

axes[1].plot(r_plot.cpu(), f_true(r_plot).cpu(), label='f true')
axes[1].plot(r_plot.cpu(), g_true(r_plot).cpu(), label='g true')
axes[1].plot(r_plot.cpu(), run_ext['f_pred'].cpu(), '--', label='f + variants')
axes[1].plot(r_plot.cpu(), run_ext['g_pred'].cpu(), ':', label='g + variants')
axes[1].set_title('Base + WL variants')
axes[1].legend(fontsize=8)

axes[2].plot(run_base['history']['metric'], label='base metric')
axes[2].plot(run_ext['history']['metric'], label='+ variants metric')
axes[2].plot(run_ext['history']['variant'], label='variant loss')
axes[2].set_title('Loss comparison')
axes[2].legend(fontsize=8)

axes[3].plot(ell.cpu(), wlv1_obs.cpu(), label='WLv1 true')
axes[3].plot(ell.cpu(), run_base['wlv1_pred'].cpu(), '--', label='base pred')
axes[3].plot(ell.cpu(), run_ext['wlv1_pred'].cpu(), ':', label='+ variants pred')
axes[3].set_title('Wilson loop variant 1')
axes[3].legend(fontsize=8)

axes[4].plot(ell.cpu(), wlv2_obs.cpu(), label='WLv2 true')
axes[4].plot(ell.cpu(), run_base['wlv2_pred'].cpu(), '--', label='base pred')
axes[4].plot(ell.cpu(), run_ext['wlv2_pred'].cpu(), ':', label='+ variants pred')
axes[4].set_title('Wilson loop variant 2')
axes[4].legend(fontsize=8)

axes[5].remove()
ax3d = fig.add_subplot(2, 3, 6, projection='3d')
ax3d.plot_surface(X.numpy(), Y.numpy(), Z.numpy(), cmap='viridis', linewidth=0, antialiased=True, alpha=0.9)
ax3d.set_title('Bulk from slices')
ax3d.set_xlabel('X')
ax3d.set_ylabel('Y')
ax3d.set_zlabel('r')

plt.suptitle('AdS4 Phase-Lock with WL Variants (v7)')
plt.tight_layout()
plt.savefig('ads4_phase_lock_v7_wl_variants.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v7_wl_variants.png')


In [ ]:
# Error diagnostics for extended run
f_true_vals = f_true(r_plot).detach()
g_true_vals = g_true(r_plot).detach()
f_err = (run_ext['f_pred'] - f_true_vals).abs()
g_err = (run_ext['g_pred'] - g_true_vals).abs()
f_rel = f_err / (f_true_vals.abs() + 1e-6)
g_rel = g_err / (g_true_vals.abs() + 1e-6)

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(r_plot.cpu(), f_err.cpu(), label='f abs err')
plt.plot(r_plot.cpu(), g_err.cpu(), label='g abs err')
plt.title('absolute error')
plt.xlabel('r')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(r_plot.cpu(), f_rel.cpu(), label='f rel err')
plt.plot(r_plot.cpu(), g_rel.cpu(), label='g rel err')
plt.title('relative error')
plt.xlabel('r')
plt.legend()

plt.tight_layout()
plt.savefig('ads4_phase_lock_v7_wl_variants_errors.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved: ads4_phase_lock_v7_wl_variants_errors.png')
